# Your own Colab copy

Choose File, Save a copy in Drive before working. Run the setup cell once, then work through the starter. Download your notebook and exported ZIP before disconnecting. Colab files can disappear when a runtime ends. Open Grading & notes on the task page after finishing your attempt.

In [ ]:
from pathlib import Path
import hashlib
import os
import urllib.request
import zipfile

work = Path('/content/icaic-real-fake') if Path('/content').exists() else Path.cwd() / 'icaic-real-fake'
work.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')
os.environ.setdefault('SDL_AUDIODRIVER', 'dummy')
os.environ.setdefault('PYGAME_HIDE_SUPPORT_PROMPT', '1')
payload = urllib.request.urlopen('https://elteomc.github.io/icaic-practice-hub/packs/real-fake.zip', timeout=90).read()
assert hashlib.sha256(payload).hexdigest() == 'e64a86b54c728805fa4ec31476a7b6a2b733cf54e5a1ef015fc1724d903285bc', 'The task pack changed. Reopen the latest notebook from the hub.'
archive_path = work / 'task.zip'
archive_path.write_bytes(payload)
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(work)
os.chdir(work / 'real-fake')
import sys
sys.path.insert(0, str(Path.cwd()))
print('Task folder:', Path.cwd())


# Real or Fake Images

Read Statement.md before changing code. This starter checks the input and export path. It deliberately contains no trained solution. Keep your experiments and reasoning here.

In [ ]:
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
with zipfile.ZipFile('images_train.zip') as archive:
    archive.extractall('.')
train = pd.read_csv('train.csv')
print(train.groupby('label').size())
fig, axes = plt.subplots(1, 6, figsize=(9, 2))
for axis, row in zip(axes, train.iloc[:6].itertuples()):
    axis.imshow(Image.open(row.file))
    axis.set_title('Generated' if row.label else 'Real')
    axis.axis('off')
plt.show()

## Your training experiment

Split the rows into training and development sets. Load pixels as float32, reorder H,W,C to C,H,W, and normalize with pixel / 127.5 - 1. The supplied CNN meets the structure rules but has no trained weights. It returns probabilities. Add your data loader, loss, optimizer, and development check.

In [ ]:
import torch
from submission_model import MyModel
torch.set_num_threads(2)
torch.manual_seed(2026)
model = MyModel()
first = np.asarray(Image.open(train.iloc[0]['file']).convert('RGB'), dtype=np.float32)
x = torch.tensor(first.transpose(2, 0, 1) / 127.5 - 1).unsqueeze(0)
print('Input shape:', x.shape, 'output:', model(x).detach())

# Add your development split and training here


## Export for evaluation

Run this after training. If you change the architecture, update submission_model.py to match before exporting. Only the class definition and trained state dictionary belong in the ZIP. The cell also works now to establish an untrained baseline.

In [ ]:
from pathlib import Path
import zipfile
model.eval()
torch.save(model.state_dict(), 'submission_dic.pth')
with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in ['submission_model.py', 'submission_dic.pth']:
        archive.write(name, name)
print('Saved submission.zip:', Path('submission.zip').stat().st_size, 'bytes')

## Keep evidence

Record your development split, seed, result, and one next hypothesis. Export submission.zip. Grade it once to check the pipeline and use your development split to measure improvements.

## Download for the hub

Run this export cell to download your ZIP. Open Grading & notes on the task page and follow the grading instructions. If you are practicing independently, finish your attempt before checking the final practice score.

In [ ]:
from pathlib import Path
assert Path('submission.zip').is_file(), 'Run the model export cell first'
output = 'submission.zip'
try:
    from google.colab import files
    files.download(output)
except ImportError:
    print('Saved ZIP at', Path(output).resolve())
